# Нейронные сети и обработка естественного языка - NLP

# Модуль 3. Трансформеры


**Смысл ноутбука:** показать, как трансформер
1) видит зависимости (через attention),  
2) ищет связанные записи (через эмбеддинги),  
3) ловит аномалии в последовательностях (через ошибку предсказания),  
4) генерирует текст (next-token prediction).



In [ ]:

import math, random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"
device



## Как “получить зависимости” из трансформера
- **Attention-матрица** `A[i,j]` показывает, насколько позиция `i` использовала позицию `j`.
- **Связанные записи** можно искать по эмбеддингам (вектора из модели) через kNN/кластеры.
- **Аномалии** в последовательности: учим модель “норме”, а выбросы видим как большую ошибку предсказания.



---
# 1) Текст (отзывы): классификация + attention
Сделаем маленький трансформер “с нуля” (очень упрощённо):
- токенизация по пробелам,
- Embedding → self-attention блоки → pooling → классификатор,
- attention возвращаем наружу.


In [ ]:

# Игрушечные отзывы: 1=позитив, 0=негатив
train_texts = [
    "очень понравилось быстро вкусно рекомендую",
    "супер сервис вежливо и быстро",
    "отличный товар качество на высоте",
    "люблю это место всегда вкусно",
    "ужасно долго и холодно больше не приду",
    "отвратительно грубо и невкусно",
    "ненавижу ожидание сервис ужасный",
    "плохое качество сломалось сразу",
    "вкусно но долго ждал",
    "быстро но качество плохое",
    "очень вкусно и уютно",
    "грубо и медленно",
]
train_labels = [1,1,1,1, 0,0,0,0, 0,0,1,0]

test_texts = [
    "очень вкусно рекомендую",
    "сервис грубо ужасно",
    "быстро и вежливо",
    "качество отвратительно",
]
test_labels = [1,0,1,0]

def tokenize(s): return s.lower().strip().split()

special = ["<pad>", "<unk>"]
vocab = {t:i for i,t in enumerate(special)}
for s in train_texts + test_texts:
    for tok in tokenize(s):
        vocab.setdefault(tok, len(vocab))

id2tok = {i:t for t,i in vocab.items()}
pad_id = vocab["<pad>"]
unk_id = vocab["<unk>"]

def encode(s, max_len=10):
    ids = [vocab.get(t, unk_id) for t in tokenize(s)][:max_len]
    ids += [pad_id]*(max_len-len(ids))
    return ids

max_len = 10
X_train = torch.tensor([encode(s, max_len) for s in train_texts], dtype=torch.long)
y_train = torch.tensor(train_labels, dtype=torch.long)
X_test  = torch.tensor([encode(s, max_len) for s in test_texts], dtype=torch.long)
y_test  = torch.tensor(test_labels, dtype=torch.long)

class ReviewDS(Dataset):
    def __init__(self, X, y): self.X, self.y = X, y
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]

train_loader = DataLoader(ReviewDS(X_train, y_train), batch_size=4, shuffle=True)
test_loader  = DataLoader(ReviewDS(X_test,  y_test),  batch_size=4, shuffle=False)

len(vocab), X_train.shape


In [ ]:

class TinyAttentionBlock(nn.Module):
    def __init__(self, d_model=64, n_heads=4, dropout=0.1):
        super().__init__()
        self.mha = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.norm1 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_model*4),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model*4, d_model),
        )
        self.norm2 = nn.LayerNorm(d_model)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, key_padding_mask=None):
        out, attn = self.mha(
            x, x, x,
            key_padding_mask=key_padding_mask,
            need_weights=True,
            average_attn_weights=False  # [B,H,T,T]
        )
        x = self.norm1(x + self.drop(out))
        x = self.norm2(x + self.drop(self.ff(x)))
        return x, attn

class TinyTransformerReviews(nn.Module):
    def __init__(self, vocab_size, max_len, d_model=64, n_heads=4, n_layers=2, n_classes=2, dropout=0.1):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, d_model)
        self.pos = nn.Parameter(torch.randn(1, max_len, d_model) * 0.02)
        self.blocks = nn.ModuleList([TinyAttentionBlock(d_model, n_heads, dropout) for _ in range(n_layers)])
        self.head = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, n_classes),
        )

    def forward(self, ids):
        B, T = ids.shape
        x = self.emb(ids) + self.pos[:, :T, :]
        pad_mask = (ids == pad_id)  # True = ignore
        attn_all = []
        for blk in self.blocks:
            x, attn = blk(x, key_padding_mask=pad_mask)
            attn_all.append(attn)
        mask = (~pad_mask).float().unsqueeze(-1)
        pooled = (x * mask).sum(dim=1) / mask.sum(dim=1).clamp_min(1.0)
        return self.head(pooled), attn_all

model_txt = TinyTransformerReviews(len(vocab), max_len=max_len).to(device)
opt = torch.optim.AdamW(model_txt.parameters(), lr=3e-3, weight_decay=1e-2)
crit = nn.CrossEntropyLoss()

def acc(model, loader):
    model.eval()
    ps, ys = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits, _ = model(xb)
            ps.append(logits.argmax(1).cpu().numpy())
            ys.append(yb.cpu().numpy())
    ps = np.concatenate(ps); ys = np.concatenate(ys)
    return float((ps==ys).mean())

for epoch in range(1, 61):
    model_txt.train()
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        logits, _ = model_txt(xb)
        loss = crit(logits, yb)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()
    if epoch in [1,2,3,5,10,20,40,60]:
        print(f"epoch {epoch:02d} | train acc {acc(model_txt, train_loader):.3f} | test acc {acc(model_txt, test_loader):.3f}")


In [ ]:

def show_attention(sentence, model, layer=-1, head=0):
    ids = torch.tensor([encode(sentence, max_len)], dtype=torch.long).to(device)
    model.eval()
    with torch.no_grad():
        logits, attn_all = model(ids)
        prob = torch.softmax(logits, dim=1)[0].cpu().numpy()
    attn = attn_all[layer][0, head].detach().cpu().numpy()  # [T,T]

    ids0 = ids[0].detach().cpu().tolist()
    real_len = ids0.index(pad_id) if pad_id in ids0 else len(ids0)
    toks = [id2tok[i] for i in ids0[:real_len]]
    A = attn[:real_len, :real_len]

    importance = A.sum(axis=0)  # сколько внимания получил токен
    order = np.argsort(-importance)

    print("Sentence:", sentence)
    print("Pred prob [neg,pos]:", prob.round(3))
    print("\nTop tokens by attention received:")
    for i in order:
        print(f"  {toks[i]:<15} {importance[i]:.3f}")
    print("\nAttention matrix (rounded):")
    print("tokens:", toks)
    print(np.round(A, 2))

show_attention("сервис был ужасно грубо", model_txt, layer=-1, head=0)


## Визуализация матрицы внимания (Attention Heatmap)

**Что это показывает:**
- **Левый график (Heatmap)**: матрица $A[i,j]$, где каждый элемент показывает, насколько сильно позиция $i$ (Query) обращает внимание на позицию $j$ (Key). Красные клетки = высокое внимание, жёлтые = низкое.
- **Правый график (Bar chart)**: суммарное внимание, полученное каждым токеном от всех остальных. Топ-3 токена выделены красным.

**Интерпретация:**
- Диагональ часто активна → модель опирается на сам токен.
- Горизонтальные полосы → один токен "притягивает" внимание многих.
- Вертикальные полосы → один токен смотрит на многие.


In [ ]:
def visualize_attention_heatmap(sentence, model, layer=-1, head=0, figsize=(10, 6)):
    """
    Визуализирует матрицу внимания (attention) в виде heatmap.
    
    Args:
        sentence: исходное предложение
        model: обученная модель трансформера
        layer: номер слоя (-1 = последний)
        head: номер головы внимания (head)
        figsize: размер фигуры (ширина, высота)
    """
    ids = torch.tensor([encode(sentence, max_len)], dtype=torch.long).to(device)
    model.eval()
    with torch.no_grad():
        logits, attn_all = model(ids)
        prob = torch.softmax(logits, dim=1)[0].cpu().numpy()
    
    attn = attn_all[layer][0, head].detach().cpu().numpy()  # [T,T]
    
    ids0 = ids[0].detach().cpu().tolist()
    real_len = ids0.index(pad_id) if pad_id in ids0 else len(ids0)
    toks = [id2tok[i] for i in ids0[:real_len]]
    A = attn[:real_len, :real_len]
    
    # Создаём фигуру с подграфиками
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=figsize)
    
    # Heatmap матрицы внимания
    sns.heatmap(A, annot=True, fmt='.2f', cmap='YlOrRd', cbar=True, 
                xticklabels=toks, yticklabels=toks, ax=ax1, 
                cbar_kws={'label': 'Attention weight'})
    ax1.set_title(f'Attention Matrix (Layer {layer}, Head {head})', fontsize=14, fontweight='bold')
    ax1.set_xlabel('Key (откуда внимание)', fontsize=12)
    ax1.set_ylabel('Query (куда внимание)', fontsize=12)
    
    # График распределения внимания по токенам
    importance = A.sum(axis=0)  # сколько внимания получил каждый токен
    colors = ['red' if i in np.argsort(-importance)[:3] else 'steelblue' for i in range(len(toks))]
    ax2.bar(range(len(toks)), importance, color=colors, alpha=0.7, edgecolor='black')
    ax2.set_xticks(range(len(toks)))
    ax2.set_xticklabels(toks, rotation=45, ha='right')
    ax2.set_ylabel('Total Attention Received', fontsize=12)
    ax2.set_title('Importance of Each Token', fontsize=14, fontweight='bold')
    ax2.grid(axis='y', alpha=0.3)
    
    plt.suptitle(f'Sentence: "{sentence}"\nPrediction: [neg, pos] = {prob.round(3)}', 
                 fontsize=12, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()
    
    return A, toks, importance

# Примеры использования:
visualize_attention_heatmap("очень понравилось вкусно рекомендую", model_txt, layer=-1, head=0)
visualize_attention_heatmap("ужасно грубо невкусно", model_txt, layer=-1, head=1)

**ПРАКТИКА**

Создайте свой "игрушечный" датасет, обучите модель и посмотрите на матрицы внимания для ваших данных.

In [ ]:
# ваш код здесь

## 2. Генерация названий несуществующих городов

Ожидается файл `geodata_cities_cyr.csv` в текущей папке ноутбука или рядом с ним.

Нас интересует колонка `locTitleLocal` — локальное название населённого пункта на кириллице.

In [ ]:
df = pd.read_csv("data/cities_cyr.csv", low_memory=False)
df

### Очищаем названия

Нам нужны именно **чистые названия**, чтобы модель училась на строках вроде:

- МОСКВА
- ТВЕРЬ
- НОВОСИБИРСК

а не на служебных хвостах, мусоре и альтернативных формах.

Ниже мы:
- берём `city_name`
- переводим в верхний регистр
- оставляем кириллицу, дефис и пробел
- отбрасываем слишком короткие и слишком длинные варианты
- убираем дубликаты

In [ ]:

raw_names = (
    df["city_name"]
    .dropna()
    .astype(str)
    .str.upper()
    .str.strip()
)

def normalize_city_name(name: str) -> str:
    name = name.replace("Ё", "Е")
    name = re.sub(r"\s+", " ", name)
    name = re.sub(r"[^А-Я \-]", "", name)
    name = name.strip(" -")
    return name

names = raw_names.map(normalize_city_name)

# Фильтр: только кириллица / пробел / дефис, разумная длина
names = names[
    names.str.fullmatch(r"[А-Я \-]+", na=False)
    & (names.str.len() >= 3)
    & (names.str.len() <= 24)
]

# Иногда в базе есть совсем экзотические служебные варианты — оставим самые обычные
names = names.drop_duplicates().sort_values().reset_index(drop=True)

print(f"Всего уникальных названий после очистки: {len(names):,}")
names.sample(20, random_state=42).tolist()

### Небольшой EDA: что у нас за города

Чисто чтобы почувствовать данные.

In [ ]:

lengths = names.str.len()

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(lengths, bins=range(lengths.min(), lengths.max() + 2), edgecolor="black")
ax.set_title("Распределение длин названий")
ax.set_xlabel("Длина")
ax.set_ylabel("Количество")
plt.show()

print("Самые частые первые буквы:")
display(names.str[0].value_counts().head(20).to_frame("count"))

print("\nПримеры:")
display(names.sample(15, random_state=1).to_frame("city"))

### Строим алфавит

Будем учить **посимвольную** модель.

Добавим специальные токены:

- `<PAD>` — пустышка для выравнивания
- `<BOS>` — начало строки
- `<EOS>` — конец строки

In [ ]:

special_tokens = ["<PAD>", "<BOS>", "<EOS>"]
chars = sorted(set("".join(names.tolist())))
vocab = special_tokens + chars

stoi = {ch: i for i, ch in enumerate(vocab)}
itos = {i: ch for ch, i in stoi.items()}

PAD_ID = stoi["<PAD>"]
BOS_ID = stoi["<BOS>"]
EOS_ID = stoi["<EOS>"]

vocab_size = len(vocab)
max_name_len = int(names.str.len().max())
block_size = max_name_len + 2  # BOS + city + EOS

print("Размер словаря:", vocab_size)
print("Максимальная длина названия:", max_name_len)
print("Длина последовательности (block_size):", block_size)
print(vocab)

### Кодирование строк

Превращаем строку в последовательность индексов.

Пример:

`МОСКВА`  
→ `<BOS> М О С К В А <EOS> ... <PAD>`

In [ ]:

def encode_text(text: str):
    return [stoi[ch] for ch in text]

def decode_tokens(token_ids):
    chars_out = []
    for tid in token_ids:
        ch = itos[int(tid)]
        if ch in ("<PAD>", "<BOS>"):
            continue
        if ch == "<EOS>":
            break
        chars_out.append(ch)
    return "".join(chars_out)

def encode_city(city: str, block_size: int = block_size):
    ids = [BOS_ID] + encode_text(city) + [EOS_ID]
    if len(ids) > block_size:
        ids = ids[:block_size]
        ids[-1] = EOS_ID
    else:
        ids += [PAD_ID] * (block_size - len(ids))
    return ids

sample_city = names.iloc[0]
sample_ids = encode_city(sample_city)
print(sample_city)
print(sample_ids)
print(decode_tokens(sample_ids))

### Готовим датасет для next-token prediction

Как учится CharGPT:

- на входе: `<BOS> М О С К`
- правильный ответ: `М О С К В`

То есть модель на каждом шаге угадывает **следующий символ**.

In [ ]:

class CityDataset(Dataset):
    def __init__(self, city_names, block_size):
        self.city_names = list(city_names)
        self.block_size = block_size

    def __len__(self):
        return len(self.city_names)

    def __getitem__(self, idx):
        city = self.city_names[idx]
        ids = encode_city(city, self.block_size)
        x = torch.tensor(ids[:-1], dtype=torch.long)
        y = torch.tensor(ids[1:], dtype=torch.long)
        return x, y, city

all_names = names.tolist()
random.Random(42).shuffle(all_names)

split = int(len(all_names) * 0.95)
train_names = all_names[:split]
valid_names = all_names[split:]

train_ds = CityDataset(train_names, block_size)
valid_ds = CityDataset(valid_names, block_size)

train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)
valid_loader = DataLoader(valid_ds, batch_size=256, shuffle=False)

len(train_ds), len(valid_ds)

### positional encoding

У self-attention есть одна важная проблема:

> если просто дать модели набор символов, она видит **кто есть кто**,  
> но не очень понимает, **кто на каком месте стоит**.

Для слова `ТОМСК` порядок критичен.  
`КСМТО` — это уже что-то очень тревожное.

Поэтому к эмбеддингам символов добавляют **positional encoding** — численное представление позиции символа в строке.

### Синусоидальный positional encoding

Используем классическую формулу из оригинальной статьи про Transformer:

\[
PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i / d_{model}}}\right)
\]

\[
PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i / d_{model}}}\right)
\]

Интуитивно:

- каждая позиция получает свой «рисунок»
- близкие позиции похожи, но не одинаковы
- модель может учитывать порядок без рекуррентных сетей

### Реализуем Transformer-блок с доступом к attention weights

Чтобы построить heatmap, нам нужна не только модель, но и сами **веса внимания**.
Поэтому реализуем свой небольшой decoder-only трансформер.

In [ ]:

class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))  # (1, max_len, d_model)

    def forward(self, x):
        # x: (B, T, C)
        T = x.size(1)
        return x + self.pe[:, :T, :]


class CausalSelfAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, dropout: float, block_size: int):
        super().__init__()
        assert d_model % n_heads == 0

        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads

        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)

        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        mask = torch.tril(torch.ones(block_size, block_size))
        self.register_buffer("causal_mask", mask.view(1, 1, block_size, block_size))

    def forward(self, x, need_weights=False):
        B, T, C = x.shape

        q = self.q_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)  # (B, H, T, D)
        k = self.k_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)  # (B, H, T, T)
        att = att.masked_fill(self.causal_mask[:, :, :T, :T] == 0, float("-inf"))
        att_weights = F.softmax(att, dim=-1)
        att_weights = self.attn_dropout(att_weights)

        y = att_weights @ v  # (B, H, T, D)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.resid_dropout(self.out_proj(y))

        if need_weights:
            return y, att_weights
        return y, None


class TransformerBlock(nn.Module):
    def __init__(self, d_model: int, n_heads: int, dropout: float, block_size: int, ff_mult: int = 4):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads, dropout, block_size)
        self.ln2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, ff_mult * d_model),
            nn.GELU(),
            nn.Linear(ff_mult * d_model, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x, need_weights=False):
        attn_out, attn_weights = self.attn(self.ln1(x), need_weights=need_weights)
        x = x + attn_out
        x = x + self.ff(self.ln2(x))
        return x, attn_weights


class TinyCharGPT(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        block_size: int,
        d_model: int = 128,
        n_heads: int = 4,
        n_layers: int = 3,
        dropout: float = 0.1,
    ):
        super().__init__()
        self.block_size = block_size
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_enc = PositionalEncoding(d_model, max_len=block_size)
        self.dropout = nn.Dropout(dropout)

        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, n_heads, dropout, block_size)
            for _ in range(n_layers)
        ])

        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size)

    def forward(self, idx, targets=None, return_attn=False):
        B, T = idx.shape
        assert T <= self.block_size, "Слишком длинная последовательность"

        x = self.token_emb(idx)
        x = self.pos_enc(x)
        x = self.dropout(x)

        all_attn = []
        for block in self.blocks:
            x, attn = block(x, need_weights=return_attn)
            if return_attn:
                all_attn.append(attn)

        x = self.ln_f(x)
        logits = self.head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.reshape(-1, logits.size(-1)),
                targets.reshape(-1),
                ignore_index=PAD_ID,
            )

        if return_attn:
            return logits, loss, all_attn
        return logits, loss

### Почему Q, K и V — это не магия, а очень человеческая идея

Самое человеческое объяснение такое:

- **Q (Query)** — что я сейчас ищу
- **K (Key)** — что умеет/представляет другой символ
- **V (Value)** — какую полезную информацию он передаст

Когда модель смотрит на текущую букву, она как будто спрашивает:

> «Какие прошлые буквы сейчас для меня важны?»

Например, в слове `НОВОСИБИРСК` текущая позиция может сильнее смотреть на:
- начало слова
- соседние буквы
- часто встречающиеся куски вроде `СК`, `ОВ`, `ИН`, `АР`, `ГР`

### Очень грубо:
- Q — вопрос
- K — ярлык
- V — содержимое ответа

Дальше происходит:

1. сравнили Q со всеми K  
2. поняли, кому верить сильнее  
3. взяли смешанную сумму V  
4. получили контекст для следующего символа

Именно это потом и рисуется на heatmap внимания.

### Инициализируем модель

In [ ]:

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device =", device)

model = TinyCharGPT(
    vocab_size=vocab_size,
    block_size=block_size - 1,  # потому что x = ids[:-1]
    d_model=128,
    n_heads=4,
    n_layers=3,
    dropout=0.1,
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-2)
sum(p.numel() for p in model.parameters()) / 1e6

### Функции обучения и валидации

In [ ]:

@torch.no_grad()
def evaluate(model, loader, max_batches=None):
    model.eval()
    losses = []

    for i, (x, y, _) in enumerate(loader):
        if max_batches is not None and i >= max_batches:
            break
        x = x.to(device)
        y = y.to(device)
        _, loss = model(x, y)
        losses.append(loss.item())

    return float(np.mean(losses))


def train_one_epoch(model, loader):
    model.train()
    losses = []

    for x, y, _ in loader:
        x = x.to(device)
        y = y.to(device)

        _, loss = model(x, y)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        losses.append(loss.item())

    return float(np.mean(losses))

### Обучаем модель

Для первого прогона можно начать с 8–15 эпох.  
Если захочется более красивых названий — увеличить число эпох.

> На CPU будет медленнее. На GPU — заметно быстрее.

In [ ]:

EPOCHS = 12

history = {"train_loss": [], "valid_loss": []}

for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch(model, train_loader)
    valid_loss = evaluate(model, valid_loader)

    history["train_loss"].append(train_loss)
    history["valid_loss"].append(valid_loss)

    print(
        f"Epoch {epoch:02d} | "
        f"train_loss={train_loss:.4f} | "
        f"valid_loss={valid_loss:.4f}"
    )

### График обучения

In [ ]:

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(history["train_loss"], label="train")
ax.plot(history["valid_loss"], label="valid")
ax.set_title("Кривая обучения")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.legend()
plt.show()

### Генерация названий

Теперь научим модель придумывать **несуществующие** города.

#### Что регулирует генерацию

##### Temperature
Температура управляет степенью случайности:

- `0.3` — очень осторожно, почти без фантазии
- `0.7` — обычно приятно
- `1.0` — стандартно
- `1.3` — начинается творческий беспредел

##### Top-k
Оставляем только `k` самых вероятных символов.  
Это снижает шанс, что модель внезапно родит «ЪЪЩЙЖ».

In [ ]:

@torch.no_grad()
def sample_next_token(logits, temperature=1.0, top_k=None):
    logits = logits / max(temperature, 1e-6)

    if top_k is not None:
        values, _ = torch.topk(logits, k=min(top_k, logits.size(-1)))
        min_keep = values[..., -1, None]
        logits = torch.where(logits < min_keep, torch.full_like(logits, float("-inf")), logits)

    probs = F.softmax(logits, dim=-1)
    next_id = torch.multinomial(probs, num_samples=1)
    return next_id


@torch.no_grad()
def generate_city(
    model,
    start_letter: str,
    max_new_tokens: int = 20,
    temperature: float = 0.8,
    top_k: int | None = 10,
):
    model.eval()

    start_letter = start_letter.upper().replace("Ё", "Е")
    if start_letter not in stoi:
        raise ValueError(f"Символ {start_letter!r} отсутствует в словаре")

    ids = [BOS_ID, stoi[start_letter]]
    x = torch.tensor(ids, dtype=torch.long, device=device).unsqueeze(0)

    for _ in range(max_new_tokens):
        x_cond = x[:, -(block_size - 1):]
        logits, _ = model(x_cond)
        next_logits = logits[:, -1, :]
        next_id = sample_next_token(next_logits, temperature=temperature, top_k=top_k)

        x = torch.cat([x, next_id], dim=1)

        if next_id.item() == EOS_ID:
            break

    return decode_tokens(x[0].tolist())


def generate_many(model, letter, n=20, **kwargs):
    out = []
    for _ in range(n):
        out.append(generate_city(model, letter, **kwargs))
    return out

### Пробуем генерацию по первой букве

In [ ]:

for letter in ["М", "К", "Т", "С", "В"]:
    print(f"\nБуква {letter}:")
    for city in generate_many(model, letter, n=10, temperature=1.5, top_k=10):
        print(" -", city)

### Фильтруем реальные города и оставляем только вымышленные

Чтобы не «играть в города» против честно украденной Москвы,
будем отбрасывать названия, которые уже есть в обучающем наборе.

In [ ]:

known_cities = set(names.tolist())

def generate_fictional_city(
    model,
    start_letter: str,
    max_attempts: int = 100,
    min_len: int = 4,
    max_len: int = 18,
    **kwargs,
):
    start_letter = start_letter.upper().replace("Ё", "Е")

    for _ in range(max_attempts):
        city = generate_city(model, start_letter, **kwargs)
        if (
            city not in known_cities
            and city.startswith(start_letter)
            and min_len <= len(city) <= max_len
            and re.fullmatch(r"[А-Я \-]+", city)
        ):
            return city

    return None

for letter in ["А", "Б", "В", "Г", "Д", "К", "М", "С", "Т"]:
    print(letter, "->", generate_fictional_city(model, letter, temperature=1.5, top_k=12))

### Визуализируем attention как heatmap

Сейчас мы увидим, на какие символы модель смотрит при генерации названия.

Это самый красивый момент ноутбука:  
математика внезапно становится картинкой.

In [ ]:

@torch.no_grad()
def get_attention_maps(model, text: str):
    model.eval()
    ids = [BOS_ID] + encode_text(text)
    x = torch.tensor(ids, dtype=torch.long, device=device).unsqueeze(0)

    logits, loss, attn_layers = model(x, return_attn=True)
    return ids, attn_layers


def plot_attention_heatmap(model, text: str, layer: int = -1, head: int = 0, figsize=(8, 6)):
    ids, attn_layers = get_attention_maps(model, text)
    att = attn_layers[layer][0, head].detach().cpu().numpy()  # (T, T)

    labels = ["<BOS>"] + list(text)

    fig, ax = plt.subplots(figsize=figsize)
    im = ax.imshow(att, aspect="auto")
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels)
    ax.set_yticks(range(len(labels)))
    ax.set_yticklabels(labels)
    ax.set_title(f"Attention heatmap | layer={layer} | head={head} | text={text}")
    ax.set_xlabel("Куда смотрим")
    ax.set_ylabel("Кто смотрит")
    plt.colorbar(im, ax=ax)
    plt.show()

### Примеры heatmap

In [ ]:

plot_attention_heatmap(model, "МОСКВА", layer=-1, head=0)
plot_attention_heatmap(model, "НОВОСИБИРСК", layer=-1, head=1)
plot_attention_heatmap(model, "ТАГАНРОГ", layer=-1, head=2)

### Как читать heatmap

Строка — это символ, который **смотрит**.  
Столбец — это символ, **на который смотрят**.

В decoder-only модели используется **causal mask**, поэтому символ не может подглядывать в будущее.

То есть:
- буква на позиции 5 может смотреть на 0..5
- но не может смотреть на 6, 7, 8...

Именно поэтому на heatmap виден характерный **треугольник**.

**ПРАКТИКА**

Обучите свой собственный генератор названий (например, лекарств), физических, юридических или медицинских терминов, имен и т.д. 

HAVE FUN!


In [ ]:
# ваш код здесь